# Predicting Smartphone Addiction
Playground Series - Season 6 Episode 8

Predicting the smartphone addiction helps identify students whose digital habits may be putting them at risk, allowing earlier intervention and support. By modeling patterns across behaviors like screen time, sleep hours, and stress level, we can uncover which lifestyle factors most strongly contribute to unhealthy usage. These insights enable more targeted wellness strategies and help institutions better understand how daily routines shape long‑term digital well‑being.

Yao Yan, Walter Reade, Elizabeth Park. Predicting Smartphone Addiction. https://kaggle.com/competitions/playground-series-s6e8, 2026. Kaggle.

## About the Data


The data consists of `691369` students generated from the [Smartphone Addiction Prediction Dataset](https://www.kaggle.com/datasets/algozee/smartphone-addiction-prediction-data). Each record represents a describing lifestyle behaviors like screen time, sleep hours, and stress level. The dataset includes 12 feature columns and `1` target column `addicted_label`.

| **Field** | **Description** |
| --- | --- |
| **age** | Student’s age in years. |
| **daily_screen_time_hours** | Total screen time per day across all devices. |
| **social_media_hours** | Hours spent on social media daily. |
| **gaming_hours** | Hours spent gaming each day. |
| **work_study_hours** | Time spent on work or studying per day. |
| **sleep_hours** | Total hours of sleep per night. |
| **notifications_per_day** | Number of notifications received daily. |
| **app_opens_per_day** | How many times apps are opened per day. |
| **weekend_screen_time** | Total screen time accumulated over the weekend. |
| **gender** | Reported gender category. |
| **stress_level** | Self‑reported stress rating. |
| **academic_work_impact** | Whether screen habits affect academic performance. |
| **addicted_label** | Target variable indicating digital addiction (0/1). |

In [17]:
# imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import optuna

# Jupyter cell magic
try:
    from IPython.core.magic import register_cell_magic
    @register_cell_magic
    def skip(line, cell): return
except:
    pass

## Data Ingestion

In [3]:
season = "s6"
episode = "e8"

base_path = f"/kaggle/input/competitions/playground-series-{season}{episode}"

train = pd.read_csv(f"{base_path}/train.csv", index_col="id")
test = pd.read_csv(f"{base_path}/test.csv", index_col="id")
submission_sample = pd.read_csv(f"{base_path}/sample_submission.csv")

In [4]:
def downcasting(data: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    mem_before = data.memory_usage().sum() / 1024**2
    if verbose:
        print(f"Memory usage of dataframe is {mem_before:.2f} MB")
    
    for col in data.select_dtypes(include=["number"]).columns:
        if pd.api.types.is_integer_dtype(data[col]):
            data[col] = pd.to_numeric(data[col], downcast="integer")
        elif pd.api.types.is_float_dtype(data[col]):
            data[col] = pd.to_numeric(data[col], downcast="float")
    
    mem_after = data.memory_usage().sum() / 1024**2
    if verbose:
        print(f"Memory usage after optimization is: {mem_after:.2f} MB")
        print(f"Decreased by {(100 * (mem_before - mem_after) / mem_before):.1f}%\n")
    
    return data

print("Train train:")
train = downcasting(train)
print("Test train:")
test = downcasting(test)

Train train:
Memory usage of dataframe is 73.85 MB
Memory usage after optimization is: 45.49 MB
Decreased by 38.4%

Test train:
Memory usage of dataframe is 29.39 MB
Memory usage after optimization is: 19.22 MB
Decreased by 34.6%



## EDA

In [8]:
df = train.copy()

df.info()
display(df.describe())
display(df.head())

target = 'addicted_label'
features = [col for col in train.columns if col not in [target]]
cat_cols = train[features].select_dtypes(include=['object']).columns.tolist()
num_cols = train[features].select_dtypes(exclude=['object']).columns.tolist()

<class 'pandas.core.frame.DataFrame'>
Index: 691369 entries, 0 to 691368
Data columns (total 13 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   age                      662440 non-null  float32
 1   daily_screen_time_hours  595515 non-null  float32
 2   social_media_hours       557374 non-null  float32
 3   gaming_hours             564548 non-null  float32
 4   work_study_hours         639851 non-null  float32
 5   sleep_hours              646889 non-null  float32
 6   notifications_per_day    623785 non-null  float32
 7   app_opens_per_day        610659 non-null  float32
 8   weekend_screen_time      579306 non-null  float32
 9   gender                   662335 non-null  object 
 10  stress_level             636221 non-null  object 
 11  academic_work_impact     647145 non-null  object 
 12  addicted_label           691369 non-null  int8   
dtypes: float32(9), int8(1), object(3)
memory usage: 45.5+ MB


,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,addicted_label
count,662440.000000,595515.000000,557374.000000,564548.000000,639851.000000,646889.000000,623785.000000,610659.000000,579306.000000,691369.000000
mean,26.615406,7.640866,2.471038,1.459265,2.366971,6.804334,145.894928,102.636780,9.479865,0.709424
std,5.153387,2.721083,1.316140,0.934503,1.258591,1.234199,65.910217,48.085812,2.855701,0.454028
min,18.000000,0.500000,0.000000,0.000000,0.000000,4.500000,20.000000,15.000000,0.510000,0.000000
25%,22.000000,5.480000,1.450000,0.700000,1.360000,5.780000,93.000000,64.000000,7.280000,0.000000
50%,27.000000,7.770000,2.310000,1.330000,2.200000,6.800000,150.000000,104.000000,9.580000,1.000000
75%,31.000000,9.840000,3.370000,2.090000,3.200000,7.870000,204.000000,145.000000,11.750000,1.000000
max,35.000000,15.000000,8.000000,4.000000,6.000000,9.000000,250.000000,180.000000,17.559999,1.000000


,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
id,,,,,,,,,,,,,
0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


In [7]:
# ============================
# Missing Data
# ============================
missing_counts = df.isnull().sum()
missing_percent = (df.isnull().mean() * 100).round(2)

missing_df = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent
})

print("\nMissing Data Summary:")
display(missing_df.sort_values("missing_percent", ascending=False))


Missing Data Summary:


,missing_count,missing_percent
social_media_hours,133995,19.38
gaming_hours,126821,18.34
weekend_screen_time,112063,16.21
daily_screen_time_hours,95854,13.86
app_opens_per_day,80710,11.67
notifications_per_day,67584,9.78
stress_level,55148,7.98
work_study_hours,51518,7.45
sleep_hours,44480,6.43
academic_work_impact,44224,6.40


## Preprocessing

Handling missing values:
- Continuous variables were imputed using median substitution, ensuring robustness against skewed distributions
- Categorical fields were imputed using the most‑frequent category, preserving dominant class structure without introducing synthetic categories.

All categorical attributes were label‑encoded to convert nominal values into integer representations. This encoding approach maintains deterministic mappings and is well‑suited for tree‑based learners.

In [15]:
target = 'addicted_label'

X = train.drop(target, axis=1)
y = train[target]

In [16]:
num_imputer = SimpleImputer(strategy='median')
train[num_cols] = num_imputer.fit_transform(X[num_cols])
test[num_cols] = num_imputer.transform(test[num_cols])

cat_imputer = SimpleImputer(strategy='most_frequent')
train[cat_cols] = cat_imputer.fit_transform(X[cat_cols])
test[cat_cols] = cat_imputer.transform(test[cat_cols])

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(X[col])
    test[col] = le.transform(test[col])
    label_encoders[col] = le

## Modelling

XGBoost was chosen because gradient‑boosted trees consistently excel on structured tabular data and can model complex non‑linear interactions across behavioral features such as screen time, notifications, and social media activity. Its regularization and boosting framework allow it to learn subtle patterns without overfitting. Combined with robust preprocessing and Stratified K‑Fold cross‑validation, XGBoost provides a stable, high‑performing approach.

In [21]:
%%skip
def objective_xgb(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "verbosity": 0,
        "random_state": 42,

        # Core hyperparameters
        "n_estimators": trial.suggest_int("n_estimators", 200, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15),
        "max_depth": trial.suggest_int("max_depth", 4, 10),

        # Sampling
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

        # Regularization
        "gamma": trial.suggest_float("gamma", 0, 3),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 2.0),
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, valid_idx in skf.split(X, y):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model = xgb.XGBClassifier(**params)
        model.fit(X_train, y_train)

        probs = model.predict_proba(X_valid)[:, 1]

        score = roc_auc_score(y_valid, probs)
        scores.append(score)

    return np.mean(scores)


study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=30)
best_xgb_params = study_xgb.best_params
print(best_xgb_params)

In [22]:
best_xgb_params = {'n_estimators': 498, 'learning_rate': 0.12060744952816418, 'max_depth': 8, 'subsample': 0.8848706555451172, 'colsample_bytree': 0.679603003322077, 'gamma': 0.7318089027675685, 'reg_lambda': 3.9882866674445516, 'reg_alpha': 0.3267330870773333}

In [23]:
best_params = best_xgb_params.copy()
best_params.update({
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "verbosity": 0,
    "random_state": 42
})

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

test_preds = np.zeros(len(test))

for train_idx, valid_idx in skf.split(X, y):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = xgb.XGBClassifier(**best_params)
    model.fit(X_train, y_train)

    test_preds += model.predict_proba(test)[:, 1] / skf.n_splits

print(test_preds)

[0.99921782 0.93900397 0.94153787 ... 0.16894635 0.79072218 0.81260993]


## Submission

In [28]:
submission = pd.DataFrame({
    'id': test.index, 
    'addicted_label': test_preds
})
submission.to_csv('submission.csv', index=False)